In [1]:
import os
import sys

for _ in range(4):
    os.chdir("..")

# os.getcwd()

In [2]:
RANDOM_SEED = 42
PROJECT_PATH = './projects/main'
SOURCE_FILENAME = 'dayuses.csv'

### Imports

In [3]:
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
import scipy.stats as ss
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor, CatBoostClassifier, monoforest
from matplotlib import pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

In [4]:
%run projects/scripts/reload

### Source file

\- Quick look at raw CSV  
Same anonymized day-level user activity log (~49.4M rows, 830K users).

In [5]:
pd.read_csv(PROJECT_PATH + '/source/' + SOURCE_FILENAME)

,event_date,search,cat,has_search_to_cart,has_search_to_ord,has_cat_to_cart,has_cat_to_ord,search_to_cart,search_to_ord,cat_to_cart,cat_to_ord,gmv_search,gmv_cat,to_cart,to_ord,gmv,searches,user_id
0,2025-01-01,1,1,1,0,1,0,1,0,1,0,0.000000,0.0,2,0,0.000000,3,704555
1,2025-01-01,1,0,0,0,0,0,0,0,0,0,0.000000,0.0,0,0,0.000000,2,588098
2,2025-01-01,1,1,0,0,0,0,0,0,0,0,0.000000,0.0,0,0,0.000000,1,257110
3,2025-01-01,1,0,1,1,0,0,2,2,0,0,53.693340,0.0,2,2,53.693340,2,249640
4,2025-01-01,1,0,1,1,0,0,3,3,0,0,723.934309,0.0,3,3,723.934309,7,151379
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49414699,2025-10-31,0,0,0,0,0,0,0,0,0,0,0.000000,0.0,0,0,0.000000,0,259179
49414700,2025-10-31,1,0,0,0,0,0,0,0,0,0,0.000000,0.0,0,0,0.000000,7,365839
49414701,2025-10-31,0,1,0,0,0,0,0,0,0,0,0.000000,0.0,0,0,0.000000,0,131933
49414702,2025-10-31,1,0,1,0,0,0,1,0,0,0,0.000000,0.0,1,0,0.000000,1,671156


# 0. Data Loading and Processing

\- Required columns for validation

In [6]:
required_columns = [
    'search', 'cat',
    'has_search_to_cart', 'has_search_to_ord',
    'has_cat_to_cart', 'has_cat_to_ord',
    'search_to_cart', 'search_to_ord',
    'cat_to_cart', 'cat_to_ord',
    'gmv_search', 'gmv_cat',
    'to_cart', 'to_ord', 'gmv', 'searches'
]

\- Initialize and load SessionProcessor

In [7]:
USER_ID_COL = 'user_id'
session_processor_kwargs = {
    'timestamp_col': 'event_date',
    'user_id_col': USER_ID_COL,
    'required_columns': required_columns # optional check
}
session_processor = SessionProcessor(**session_processor_kwargs)
session_processor.load(file_path=PROJECT_PATH + '/source/' + SOURCE_FILENAME)

,event_date,search,cat,has_search_to_cart,has_search_to_ord,has_cat_to_cart,has_cat_to_ord,search_to_cart,search_to_ord,cat_to_cart,cat_to_ord,gmv_search,gmv_cat,to_cart,to_ord,gmv,searches,user_id
0,2025-01-01,1,1,1,0,1,0,1,0,1,0,0.000000,0.0,2,0,0.000000,3,704555
70093,2025-01-01,1,0,0,0,0,0,0,0,0,0,0.000000,0.0,0,0,0.000000,1,226882
70092,2025-01-01,1,1,0,0,0,0,0,0,0,0,0.000000,0.0,0,0,0.000000,2,198016
70091,2025-01-01,1,1,1,1,0,0,1,1,0,0,57.034206,0.0,1,1,57.034206,4,453919
70090,2025-01-01,0,1,0,0,0,0,0,0,0,0,0.000000,0.0,0,0,0.000000,0,10681
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49283909,2025-10-31,1,1,1,1,0,0,2,2,0,0,75.776274,0.0,2,2,75.776274,4,530266
49283910,2025-10-31,1,0,1,0,0,0,1,0,0,0,0.000000,0.0,1,0,0.000000,1,475856
49283911,2025-10-31,0,0,0,0,0,0,0,0,0,0,0.000000,0.0,0,0,0.000000,0,517508
49283902,2025-10-31,1,0,1,1,0,0,1,1,0,0,15.485823,0.0,1,1,15.485823,4,280717


\- Total unique users in the dataset

Confirms we have ~830K unique users.

In [8]:
session_processor.get_users_power()

829468

## 0.1 Feature Engineering

\- Define aggregation rules (MapReduce pattern)

Identical to the orbital notebook: same MapReduceRules for day-level  
and 30-day period-level aggregation (frequency, recency, gmv, etc.)  
(See orbital notebook correspondig cells for detailed comments on each rule.)

In [9]:
parse_session = [
    MapReduceRule(lambda row: 1 * (row['search'] + row['cat'] > 0), 'max', alias='is_active'),
    MapReduceRule('gmv', 'sum'),
    MapReduceRule('search', 'max'),
    MapReduceRule('cat', 'max'),
    MapReduceRule(lambda row: 1 * (row['has_search_to_cart'] + row['has_cat_to_cart'] > 0), 'max', alias='has_to_cart'),
    MapReduceRule(lambda row: 1 * (row['has_search_to_ord'] + row['has_cat_to_ord'] > 0), 'max', alias='has_to_ord'),
    MapReduceRule(lambda row: row['search_to_cart'] + row['cat_to_cart'], 'sum', alias='to_cart'),
    MapReduceRule(lambda row: row['search_to_ord'] + row['cat_to_ord'], 'sum', alias='to_ord'),
]


parse_discrete_events = [
    MapReduceRule(
        'is_active',
        'sum', alias='frequency'
    ), 
    MapReduceRule(
        lambda row: row['period'] if row['is_active'] > 0 else None,
        'max', alias='recency'
    ), 
    MapReduceRule(
        'gmv', 'sum'
    ),
    MapReduceRule('search', 'sum'),
    MapReduceRule('cat', 'sum'),
    MapReduceRule('has_to_cart', 'sum'),
    MapReduceRule('has_to_ord', 'sum'),
    MapReduceRule('to_cart', 'sum'),
    MapReduceRule('to_ord', 'sum'),
]

# Feature rule: 30-day rolling window, daily granularity
cmplx_feature_rule = AggregationRule(
    freq='D',
    periods = 30,
    parse_sessions=parse_session,
    parse_discrete_events=parse_discrete_events,
)

# Target rule: 30-day forward GMV (= lLTV)
m_target_rule = AggregationRule(
    freq='D',
    periods = 30,
    parse_sessions=[MapReduceRule('gmv', 'sum')],
    parse_discrete_events=[MapReduceRule('gmv', 'sum')],
)

### Example: standalone SessionAggregator (for illustration)

Shows how a single aggregation call works before using the full pipeline.

# 1. Collect Learning Data


\- Initialize FeatureExtractor

#### Save/load checkpoint

\- Load pre-computed learning data from CSV

Skips the expensive aggregation step by loading the artifact   
produced in orbital notebook

In [10]:
learn_data_file_name = '2025-03-12.csv'
learn_data_file_path = PROJECT_PATH + '/artifacts/learning/data/' + learn_data_file_name

# Save
# learn_data.reset_index().to_csv(learn_data_file_path, index = False)

# Load
learn_data = pd.read_csv(learn_data_file_path)
learn_data['timestamp'] = pd.to_datetime(learn_data['timestamp'])
learn_data.set_index(['timestamp', USER_ID_COL], inplace=True)
learn_data

frequency_0  recency_0        gmv_0  search_0  cat_0  \
timestamp  user_id                                                         
2025-03-12 704555          27.0       30.0    47.423065      23.0   22.0   
           226882          15.0       29.0    18.072995      15.0    1.0   
           198016           8.0       20.0     0.000000       8.0    2.0   
           453919          16.0       18.0  1283.582175      16.0    3.0   
           10681           15.0       29.0     0.000000      11.0   12.0   
...                         ...        ...          ...       ...    ...   
           698647           NaN        NaN          NaN       NaN    NaN   
           186142           NaN        NaN          NaN       NaN    NaN   
           637145           NaN        NaN          NaN       NaN    NaN   
           414569           NaN        NaN          NaN       NaN    NaN   
           121959           NaN        NaN          NaN       NaN    NaN   

                    has_to_cart_0  has_to_ord_0  to_cart_0  to_ord_0  \
timestamp  user_id                                                     
2025-03-12 704555            12.0           3.0       36.0       3.0   
           226882            11.0           2.0       41.0       2.0   
           198016             6.0           0.0       15.0       0.0   
           453919            12.0           9.0       36.0      16.0   
           10681             11.0           0.0       26.0       0.0   
...                           ...           ...        ...       ...   
           698647             NaN           NaN        NaN       NaN   
           186142             NaN           NaN        NaN       NaN   
           637145             NaN           NaN        NaN       NaN   
           414569             NaN           NaN        NaN       NaN   
           121959             NaN           NaN        NaN       NaN   

                    gmv_target  
timestamp  user_id              
2025-03-12 704555   286.724683  
           226882    61.421165  
           198016     5.284452  
           453919   151.992792  
           10681      5.184195  
...                        ...  
           698647          NaN  
           186142          NaN  
           637145          NaN  
           414569          NaN  
           121959          NaN  

[829468 rows x 10 columns]

## 1.1. Data Preparation for Model Training

\- Define feature and target column names

In [11]:
FEATURES = [
    'frequency_0', 'recency_0', 'gmv_0'
] + [
    'search_0', 'cat_0', 'has_to_cart_0', 'has_to_ord_0', 'to_cart_0', 'to_ord_0'
]
TARGET = 'gmv_target'

\- Data preparation (KEY DIFFERENCE from orbital notebook)

In the orbital notebook, "out" users (NaN features) were DROPPED,
and the target was log-transformed.

Here, for the predictive baseline:
   - "out" users are KEPT (NaN features remain in the data)
   - Target is NOT log-transformed (raw GMV scale)
   - inv_transform is identity: no transformation applied


In [12]:
transform = lambda t: np.log10(t + 1)     # used only for visualization
inv_transform = lambda t: t               # ← identity: NO transform on target

# (1) Keep only users with non-NaN features (exclude "out" users)
# non_out_index = learn_data[FEATURES].dropna().index
# prepared_learn_data = learn_data.loc[non_out_index].copy()

# (2) Users with features but no forward activity → target = 0
prepared_learn_data = learn_data.copy()
prepared_learn_data[TARGET] = prepared_learn_data[TARGET].fillna(0)

# (3) Log-transform the target
# prepared_learn_data[TARGET] = prepared_learn_data[TARGET].apply(transform)

# ~~(4) Exclude users with zero-target~~
# prepared_learn_data = prepared_learn_data[prepared_learn_data[TARGET] > 0]

\- Train/validation split (70/30)

(more users than orbital's 370K because "out" users included)

In [13]:
data_train, data_val = train_test_split(prepared_learn_data, test_size=0.3, random_state=RANDOM_SEED)
data_train

frequency_0  recency_0       gmv_0  search_0  cat_0  \
timestamp  user_id                                                        
2025-03-12 707678           NaN        NaN         NaN       NaN    NaN   
           401855          14.0       24.0   28.396157      11.0    7.0   
           203091           3.0       29.0    0.000000       3.0    0.0   
           381897          14.0       27.0  166.964619      14.0    0.0   
           397704          19.0       30.0  327.084560      19.0    0.0   
...                         ...        ...         ...       ...    ...   
           170287          14.0       30.0    0.000000      13.0    6.0   
           590080           9.0       30.0   22.871384       8.0    2.0   
           36808            NaN        NaN         NaN       NaN    NaN   
           8836             NaN        NaN         NaN       NaN    NaN   
           268905          13.0       29.0  125.095738      13.0    2.0   

                    has_to_cart_0  has_to_ord_0  to_cart_0  to_ord_0  \
timestamp  user_id                                                     
2025-03-12 707678             NaN           NaN        NaN       NaN   
           401855             9.0           3.0       17.0       3.0   
           203091             0.0           0.0        0.0       0.0   
           381897             2.0           2.0        2.0       2.0   
           397704            16.0           9.0       39.0      13.0   
...                           ...           ...        ...       ...   
           170287             2.0           0.0        2.0       0.0   
           590080             2.0           1.0        2.0       1.0   
           36808              NaN           NaN        NaN       NaN   
           8836               NaN           NaN        NaN       NaN   
           268905             4.0           1.0        8.0       1.0   

                    gmv_target  
timestamp  user_id              
2025-03-12 707678     0.000000  
           401855    16.058413  
           203091     0.000000  
           381897   108.380971  
           397704   367.456229  
...                        ...  
           170287     0.000000  
           590080    33.496541  
           36808      0.000000  
           8836       0.000000  
           268905     0.000000  

[580627 rows x 10 columns]

\- Prepare evaluation dataset and LearnData wrapper  
inv_transform is identity here, so data_eval target = raw GMV.

In [14]:
data_eval = (
    prepared_learn_data
    # .sample(100_000, random_state=RANDOM_SEED)
    .assign(**{TARGET: lambda row: inv_transform(row[TARGET])})
)

ldata = LearnData(FEATURES, TARGET, data_train, data_val, data_eval)
ldata._eval

frequency_0  recency_0        gmv_0  search_0  cat_0  \
timestamp  user_id                                                         
2025-03-12 704555          27.0       30.0    47.423065      23.0   22.0   
           226882          15.0       29.0    18.072995      15.0    1.0   
           198016           8.0       20.0     0.000000       8.0    2.0   
           453919          16.0       18.0  1283.582175      16.0    3.0   
           10681           15.0       29.0     0.000000      11.0   12.0   
...                         ...        ...          ...       ...    ...   
           698647           NaN        NaN          NaN       NaN    NaN   
           186142           NaN        NaN          NaN       NaN    NaN   
           637145           NaN        NaN          NaN       NaN    NaN   
           414569           NaN        NaN          NaN       NaN    NaN   
           121959           NaN        NaN          NaN       NaN    NaN   

                    has_to_cart_0  has_to_ord_0  to_cart_0  to_ord_0  \
timestamp  user_id                                                     
2025-03-12 704555            12.0           3.0       36.0       3.0   
           226882            11.0           2.0       41.0       2.0   
           198016             6.0           0.0       15.0       0.0   
           453919            12.0           9.0       36.0      16.0   
           10681             11.0           0.0       26.0       0.0   
...                           ...           ...        ...       ...   
           698647             NaN           NaN        NaN       NaN   
           186142             NaN           NaN        NaN       NaN   
           637145             NaN           NaN        NaN       NaN   
           414569             NaN           NaN        NaN       NaN   
           121959             NaN           NaN        NaN       NaN   

                    gmv_target  
timestamp  user_id              
2025-03-12 704555   286.724683  
           226882    61.421165  
           198016     5.284452  
           453919   151.992792  
           10681      5.184195  
...                        ...  
           698647     0.000000  
           186142     0.000000  
           637145     0.000000  
           414569     0.000000  
           121959     0.000000  

[829468 rows x 10 columns]

# 2. Learnig to Orbital Segmentation

1.  **Train** model to predict target. *(CatBoost)*
2.  **Extract** Polynomial Representation from (CatBoost) Decision Tree. *(Monoforest)*
3.  **Extract** Table Rule from Polynomial Representation of (CatBoost) Decision Tree. *(PolyForest)*
4.  **Compact** Table Rule to constrained number of rules. *(CompactOperator)*

## Step-by-Step

\- Artifact location tag  
Separate from the orbital model to avoid overwriting.

In [15]:
LOCATION = 'predictive'

CatBoost configuration

Same hyperparameters as the orbital model: 1000 iterations, depth=8, RMSE.  
The key difference is that the TARGET is raw GMV (not log-transformed),  
so the model learns to predict absolute monetary value.  

In [16]:
CB_CONFIG = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 8, # 6
    'random_state': RANDOM_SEED,
    'loss_function': 'RMSE',
    'eval_metric': 'R2',
    'train_dir': PROJECT_PATH + f'/artifacts/learning/info/catboost_logs/{LOCATION}',
}

RuleExtractorPipeline orchestrates: train → polyforest → table rule → compact.

In [17]:
rule_extractor_pipeline = RuleExtractorPipeline(
    learn_data=ldata,
    project_path=PROJECT_PATH,
    cb_config=CB_CONFIG
)

### Step 2.1: Train model to predict target.

In [18]:
# Step 2.1: Train model to predict target.
cb_model = rule_extractor_pipeline.train(
    data_train,
    feature_cols=FEATURES,
    target_col=TARGET,
    rel_save_path = f'/artifacts/learning/info/catboost_model/{LOCATION}.cbm',
    eval_set=(data_val[FEATURES], data_val[TARGET]),
    verbose=100,
    use_best_model=True
)

cb_model.is_fitted()

0:	learn: 0.0186540	test: 0.0157310	best: 0.0157310 (0)	total: 79.7ms	remaining: 1m 19s
100:	learn: 0.2828269	test: 0.1850730	best: 0.1853019 (90)	total: 1.52s	remaining: 13.6s
200:	learn: 0.3096251	test: 0.1837377	best: 0.1853019 (90)	total: 2.9s	remaining: 11.5s
300:	learn: 0.3301799	test: 0.1827458	best: 0.1853019 (90)	total: 4.26s	remaining: 9.9s
400:	learn: 0.3482143	test: 0.1787922	best: 0.1853019 (90)	total: 5.58s	remaining: 8.34s
500:	learn: 0.3606752	test: 0.1754019	best: 0.1853019 (90)	total: 6.92s	remaining: 6.9s
600:	learn: 0.3715757	test: 0.1730252	best: 0.1853019 (90)	total: 8.31s	remaining: 5.52s
700:	learn: 0.3823121	test: 0.1714691	best: 0.1853019 (90)	total: 9.66s	remaining: 4.12s
800:	learn: 0.3916702	test: 0.1695496	best: 0.1853019 (90)	total: 11.1s	remaining: 2.76s
900:	learn: 0.3995934	test: 0.1681475	best: 0.1853019 (90)	total: 12.5s	remaining: 1.38s
999:	learn: 0.4063789	test: 0.1661308	best: 0.1853019 (90)	total: 13.9s	remaining: 0us

bestTest = 0.1853018652
be

True